# Vector-Only Retrieval Evaluation

This notebook runs a **vector-only** retrieval evaluation against the frozen QA benchmark (`data/qa_final.jsonl`). It uses [`VectorRetriever`](../src/retrieval/retriever.py) backed by a **local FAISS index + SQLite payload cache** (via [`SQLitePayloadFaissVectorStore`](../src/retrieval/sqlite_faiss_store.py)) — **Qdrant is not used**.

No knowledge-graph, `GraphExpansion`, `GraphTraversal`, or hybrid-fusion objects appear anywhere in this notebook, and it does not import from or read `notebooks/archive/`.

See [`specs/008-vector-retrieval-eval-notebook/quickstart.md`](../specs/008-vector-retrieval-eval-notebook/quickstart.md) for the operator guide and validation scenarios A-E.

## 1. Environment setup

In [ ]:
!rm -r /content/TextMining/

In [ ]:
!git clone https://github.com/PhuongThao-2005/TextMining.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Optional: install runtime dependencies if your environment does not have them yet.
# Uncomment and run once if needed.
%pip install -q faiss-gpu sentence-transformers pandas


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/TextMining")
if not (PROJECT_ROOT / "src").exists():
    # Useful if the notebook is launched from notebooks/ (path portability).
    PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")


## 2. Configuration

Edit only this cell to change run parameters (FR-006). No `src/` edits are required.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

# --- Benchmark input ---
QA_PATH = Path("/content/drive/Shareddrives/[Text Mining] - Project/Benchmark/qa_final.jsonl")  # default; falls back per research R10

# --- Retrieval parameters ---
TOP_K_LIST = [1, 5, 10]

# Keep the canonical FAISS artifacts on Drive, but run SQLite/FAISS from local Colab disk.
# Mounted Drive is very slow for SQLite random reads and caused 40s-200s payload loads.
DRIVE_INDEX_DIR = Path("/content/drive/Shareddrives/[Text Mining] - Project/faiss_index")
LOCAL_INDEX_DIR = Path("/content/faiss_index")
USE_LOCAL_INDEX_CACHE = True
INDEX_DIR = LOCAL_INDEX_DIR if USE_LOCAL_INDEX_CACHE else DRIVE_INDEX_DIR

MODEL_NAME = "intfloat/multilingual-e5-large"
SCORE_THRESHOLD = 0.3
EXPAND_UNITS = False  # Faster eval default: avoids extra scroll SQL expansion calls.
EMBEDDER_DEVICE = "auto"  # "auto", "cuda", "mps", or "cpu"; used when DEV_HASHING=False

# --- Run controls ---
SAMPLE_LIMIT = 10  # e.g. 25 for a fast smoke test; None evaluates all eligible cases
DEV_HASHING = False  # False uses the real FAISS index; True is only for a no-index smoke test.

# --- Output (only used if the persistence cell is triggered) ---
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUT_DIR = Path("/content/drive/Shareddrives/[Text Mining] - Project/Benchmark/eval_retrieval_only")

print("Vector-only retrieval evaluation — configuration")
print(f"  QA_PATH               = {QA_PATH}")
print(f"  TOP_K_LIST            = {TOP_K_LIST}")
print(f"  DRIVE_INDEX_DIR       = {DRIVE_INDEX_DIR}")
print(f"  LOCAL_INDEX_DIR       = {LOCAL_INDEX_DIR}")
print(f"  USE_LOCAL_INDEX_CACHE = {USE_LOCAL_INDEX_CACHE}")
print(f"  INDEX_DIR             = {INDEX_DIR}")
print(f"  MODEL_NAME            = {MODEL_NAME}")
print(f"  SCORE_THRESHOLD       = {SCORE_THRESHOLD}")
print(f"  EXPAND_UNITS          = {EXPAND_UNITS}")
print(f"  EMBEDDER_DEVICE       = {EMBEDDER_DEVICE}")
print(f"  SAMPLE_LIMIT          = {SAMPLE_LIMIT}")
print(f"  DEV_HASHING           = {DEV_HASHING}")
print(f"  OUT_DIR               = {OUT_DIR}")


## 3. Load the QA benchmark (FR-011: clear error if missing)

In [ ]:
from evaluation.io_utils import read_jsonl

if not QA_PATH.exists():
    raise FileNotFoundError(
        f"qa_final.jsonl not found at {QA_PATH}. Set QA_PATH in the configuration cell to the correct location."
    )

qa_rows = list(read_jsonl(QA_PATH))
print(f"Loaded {len(qa_rows)} raw QA rows from {QA_PATH}")


## 4. Eligibility filtering (FR-003, FR-007)

Every row is classified into exactly one of: eligible / skipped (unanswerable) / skipped (missing ground truth).

This notebook now evaluates retrieval at **document** and **provision** level only (`Doc_Precision`, `Doc_Recall`, `Provision_Precision`, `Provision_Recall`). Chunk-level evaluation has been removed.


In [ ]:
from evaluation.eligibility import select_eligible_cases

eligibility_summary = select_eligible_cases(qa_rows, sample_limit=SAMPLE_LIMIT)

print("Eligibility summary")
print(f"  total_rows                   = {eligibility_summary.total_rows}")
print(f"  eligible (evaluated)          = {len(eligibility_summary.eligible)}")
print(f"  skipped_unanswerable          = {eligibility_summary.skipped_unanswerable}")
print(f"  skipped_missing_ground_truth  = {eligibility_summary.skipped_missing_ground_truth}")

if SAMPLE_LIMIT is not None:
    print(
        f"\nNote: SAMPLE_LIMIT={SAMPLE_LIMIT} was applied — evaluated "
        f"{len(eligibility_summary.eligible)} eligible case(s) capped at SAMPLE_LIMIT "
        f"(rows beyond the cap were not examined)."
    )


## 5. Build the vector retriever (FAISS + SQLite payload cache; FR-002a, FR-012)

**Qdrant is not used by this notebook.** `DEV_HASHING=True` routes to an in-memory hashing embedder/store for the smoke-test scenario; otherwise a real FAISS index directory (`INDEX_DIR`) is required.

In [ ]:
import shutil
import time


def _copy_if_needed(src: Path, dst: Path) -> None:
    """Copy src to dst only when dst is missing, older, or size differs."""
    if not src.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() and dst.stat().st_size == src.stat().st_size and dst.stat().st_mtime >= src.stat().st_mtime:
        return
    tmp = dst.with_suffix(dst.suffix + ".tmp")
    if tmp.exists():
        tmp.unlink()
    shutil.copy2(src, tmp)
    tmp.replace(dst)


def stage_faiss_index_to_local_disk(source_dir: Path, target_dir: Path) -> Path:
    """Stage FAISS artifacts from Drive to local Colab disk for fast SQLite reads."""
    required = ["index.faiss", "payloads.jsonl"]
    optional = ["payload_cache.sqlite", "id_map.json"]
    missing = [name for name in required if not (source_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing required FAISS artifact(s) in {source_dir}: {missing}")

    t0 = time.perf_counter()
    target_dir.mkdir(parents=True, exist_ok=True)
    for name in required + optional:
        _copy_if_needed(source_dir / name, target_dir / name)
    print(f"Staged FAISS artifacts to local disk: {target_dir} ({time.perf_counter() - t0:.2f}s)")
    return target_dir


if not DEV_HASHING and USE_LOCAL_INDEX_CACHE:
    INDEX_DIR = stage_faiss_index_to_local_disk(DRIVE_INDEX_DIR, LOCAL_INDEX_DIR)
    if not (INDEX_DIR / "payload_cache.sqlite").exists():
        print(
            "WARNING: payload_cache.sqlite was not present in the staged artifacts. "
            "The first load will rebuild it locally from payloads.jsonl; subsequent runs will reuse it."
        )
else:
    print(f"Skipping local index staging (DEV_HASHING={DEV_HASHING}, USE_LOCAL_INDEX_CACHE={USE_LOCAL_INDEX_CACHE}).")


In [ ]:
from retrieval.config import VectorIndexConfig
from retrieval.embeddings import HashingEmbedder, SentenceTransformerEmbedder
from retrieval.retriever import VectorRetriever
from retrieval.stores import InMemoryVectorStore

max_k = max(TOP_K_LIST)
config = VectorIndexConfig(
    embedding_model=MODEL_NAME,
    top_k=max(max_k * 3, 30),
    top_n=max_k,
    score_threshold=SCORE_THRESHOLD,
    expand_units=EXPAND_UNITS,
)

if DEV_HASHING:
    # Smoke-test path: deterministic in-memory hashing embedder/store; no FAISS index/model needed.
    embedder = HashingEmbedder()
    store = InMemoryVectorStore()
    _resolved_device = "n/a"
else:
    from retrieval.sqlite_faiss_store import SQLitePayloadFaissVectorStore

    if not (INDEX_DIR / "index.faiss").exists() or not (INDEX_DIR / "payloads.jsonl").exists():
        raise FileNotFoundError(
            f"Could not build the FAISS-backed retriever: INDEX_DIR={INDEX_DIR} is missing "
            "'index.faiss' and/or 'payloads.jsonl'.\n"
            "Either set DEV_HASHING = True for a smoke test, or point INDEX_DIR at a directory "
            "containing 'index.faiss' and 'payloads.jsonl' (see scripts/build_vector_index.py)."
        )

    store = SQLitePayloadFaissVectorStore.load(INDEX_DIR)

    # Match the archive notebook's retrieval method: explicitly resolve the
    # SentenceTransformer device before constructing the vector retriever.
    _requested_device = str(globals().get("EMBEDDER_DEVICE", "auto")).strip().lower()
    if _requested_device in ("", "auto"):
        _resolved_device = "cpu"
        try:
            import torch

            if torch.cuda.is_available():
                _resolved_device = "cuda"
            elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
                _resolved_device = "mps"
        except Exception:
            _resolved_device = "cpu"
    else:
        _resolved_device = _requested_device

    if _resolved_device == "cuda":
        try:
            import torch

            if not torch.cuda.is_available():
                print("EMBEDDER_DEVICE='cuda' requested but CUDA is unavailable; falling back to CPU.")
                _resolved_device = "cpu"
        except Exception:
            print("Could not verify CUDA availability; falling back to CPU.")
            _resolved_device = "cpu"

    print(f"Loading FAISS vector store from {INDEX_DIR}")
    print(f"Loading SentenceTransformer embedder {MODEL_NAME!r} on device={_resolved_device!r}")
    embedder = SentenceTransformerEmbedder(
        MODEL_NAME,
        query_prefix=config.query_prefix,
        passage_prefix=config.passage_prefix,
        device=_resolved_device,
    )

retriever = VectorRetriever(config=config, embedder=embedder, store=store)
print(f"Retriever ready (store={'dev_hashing' if DEV_HASHING else 'faiss'}, device={_resolved_device}).")


## 5b. Intent extraction + query decomposition (LLM/SLM, notebook-only)

Raw benchmark questions sometimes carry filler wording that doesn't help embedding similarity. This optional step calls an **OpenAI-compatible** chat-completions endpoint to rewrite each question into a concise retrieval query *before* it reaches `VectorRetriever`.

This notebook also has an optional **query decomposition** step for testing: it asks the same LLM/API key to split the original question into focused subqueries, runs vector retrieval for each subquery, then merges/deduplicates the retrieved chunks. This is purely notebook-side experimentation — **no `src/` or codebase changes**.

**Security note:** the API key is read from a Colab secret (`Tools > Secrets`, key name `INTENT_LLM_API_KEY`) or the `INTENT_LLM_API_KEY` environment variable — never hardcode a real key directly in this cell. Set `INTENT_EXTRACTION_ENABLED = False` and `QUERY_DECOMPOSITION_ENABLED = False` to skip LLM calls entirely.


In [ ]:
import os
import json
import re


def _load_api_key(var_name: str) -> str:
    """Prefer Colab secrets (Tools > Secrets) over a plaintext notebook value."""
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(var_name)
        if value:
            return value
    except Exception:
        pass
    return os.environ.get(var_name, "")


# --- Intent extraction + query decomposition (LLM/SLM) config ---
INTENT_EXTRACTION_ENABLED = False  # Faster default: bypass LLM query rewrite
QUERY_DECOMPOSITION_ENABLED = False  # Default raw vector search; set True to test LLM subquery retrieval
QUERY_DECOMPOSITION_INCLUDE_INTENT_QUERY = False  # Keep raw query default; set True to include intent query in multi-query tests
QUERY_DECOMPOSITION_MAX_SUBQUERIES = 4

# --- RRF reranking / fusion ---
RRF_ENABLED = True  # Applies to retrieval result lists; for raw single-query search, order is preserved.
RRF_K = 60
RRF_CANDIDATE_MULTIPLIER = 3  # Retrieve more candidates per query before final RRF top-k.

# --- Cross-encoder reranking ---
# Second-stage reranker: retrieve a broader candidate pool with FAISS/RRF, then let
# BAAI/bge-reranker-v2-m3 choose the best final candidates for evaluation.
CROSS_ENCODER_RERANK_ENABLED = True
CROSS_ENCODER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
CROSS_ENCODER_DEVICE = "auto"  # "auto", "cuda", "mps", or "cpu"
CROSS_ENCODER_CANDIDATE_MULTIPLIER = 5  # Retrieve at least max_k * this many candidates before reranking.
CROSS_ENCODER_BATCH_SIZE = 16
CROSS_ENCODER_MAX_LENGTH = 512

INTENT_LLM_BASE_URL = os.environ.get("INTENT_LLM_BASE_URL", "https://api.openai.com/v1")  # any OpenAI-compatible base_url
INTENT_LLM_API_KEY = _load_api_key("INTENT_LLM_API_KEY")  # same key used for intent extraction and decomposition
INTENT_LLM_MODEL_NAME = os.environ.get("INTENT_LLM_MODEL_NAME", "gpt-4o-mini")
INTENT_LLM_TEMPERATURE = 0.0

print("Intent extraction + query decomposition — configuration")
print(f"  INTENT_EXTRACTION_ENABLED              = {INTENT_EXTRACTION_ENABLED}")
print(f"  QUERY_DECOMPOSITION_ENABLED            = {QUERY_DECOMPOSITION_ENABLED}")
print(f"  QUERY_DECOMPOSITION_INCLUDE_INTENT_QUERY = {QUERY_DECOMPOSITION_INCLUDE_INTENT_QUERY}")
print(f"  QUERY_DECOMPOSITION_MAX_SUBQUERIES     = {QUERY_DECOMPOSITION_MAX_SUBQUERIES}")
print(f"  RRF_ENABLED                           = {RRF_ENABLED}")
print(f"  RRF_K                                 = {RRF_K}")
print(f"  RRF_CANDIDATE_MULTIPLIER              = {RRF_CANDIDATE_MULTIPLIER}")
print(f"  CROSS_ENCODER_RERANK_ENABLED          = {CROSS_ENCODER_RERANK_ENABLED}")
print(f"  CROSS_ENCODER_MODEL_NAME              = {CROSS_ENCODER_MODEL_NAME}")
print(f"  CROSS_ENCODER_DEVICE                  = {CROSS_ENCODER_DEVICE}")
print(f"  CROSS_ENCODER_CANDIDATE_MULTIPLIER    = {CROSS_ENCODER_CANDIDATE_MULTIPLIER}")
print(f"  CROSS_ENCODER_BATCH_SIZE              = {CROSS_ENCODER_BATCH_SIZE}")
print(f"  CROSS_ENCODER_MAX_LENGTH              = {CROSS_ENCODER_MAX_LENGTH}")
print(f"  INTENT_LLM_BASE_URL                    = {INTENT_LLM_BASE_URL}")
print(f"  INTENT_LLM_MODEL_NAME                  = {INTENT_LLM_MODEL_NAME}")
print(f"  INTENT_LLM_API_KEY set?                = {bool(INTENT_LLM_API_KEY)}")




In [ ]:
_intent_client = None

if INTENT_EXTRACTION_ENABLED or QUERY_DECOMPOSITION_ENABLED:
    if not INTENT_LLM_API_KEY:
        raise ValueError(
            "LLM preprocessing is enabled but INTENT_LLM_API_KEY is empty. "
            "Set it via a Colab secret or the INTENT_LLM_API_KEY env var, "
            "or set INTENT_EXTRACTION_ENABLED = False and QUERY_DECOMPOSITION_ENABLED = False."
        )
    from openai import OpenAI

    _intent_client = OpenAI(base_url=INTENT_LLM_BASE_URL, api_key=INTENT_LLM_API_KEY)
    print(f"LLM preprocessing ENABLED — model={INTENT_LLM_MODEL_NAME}, base_url={INTENT_LLM_BASE_URL}")
else:
    print("LLM preprocessing DISABLED — raw questions will be sent to the retriever unchanged.")



In [ ]:
INTENT_EXTRACTION_PROMPT = (
    "Bạn là bộ phận trích xuất truy vấn cho hệ thống RAG pháp luật Việt Nam. "
    "Nhiệm vụ: đọc câu hỏi thô của người dùng và viết lại thành MỘT truy vấn tìm kiếm "
    "ngắn gọn, giữ nguyên ý định và các thực thể pháp lý quan trọng (tên, luật, điều, khoản, số hiệu, ngày tháng năm,...), "
    "loại bỏ từ ngữ dư thừa, lời chào hỏi, câu dẫn nhập.\n"
    "Chỉ trả về truy vấn đã viết lại, không giải thích, không thêm dấu nháy."
)

QUERY_DECOMPOSITION_PROMPT = f"""
Bạn là bộ phận phân rã truy vấn cho hệ thống RAG pháp luật Việt Nam.
Nhiệm vụ: đọc câu hỏi gốc và tách thành tối đa {QUERY_DECOMPOSITION_MAX_SUBQUERIES} truy vấn con độc lập, ngắn gọn, phục vụ truy xuất vector.
Yêu cầu:
- Giữ nguyên các thực thể pháp lý quan trọng: tên văn bản, số hiệu, điều/khoản/điểm, ngày tháng, đối tượng, hành vi.
- Mỗi truy vấn con tập trung vào một khía cạnh/điều kiện pháp lý riêng nếu câu hỏi có nhiều ý.
- Nếu câu hỏi chỉ có một ý, trả về 1 truy vấn con.
- Không bịa thêm thông tin ngoài câu hỏi gốc.
- Chỉ trả về JSON array các string, ví dụ: ["truy vấn con 1", "truy vấn con 2"]. Không giải thích.
""".strip()


def _ordered_unique_text(values):
    seen = set()
    out = []
    for value in values:
        text = str(value or "").strip()
        key = text.lower()
        if text and key not in seen:
            seen.add(key)
            out.append(text)
    return out


def _parse_query_list(text: str) -> list[str]:
    """Parse a JSON array from the LLM; tolerate numbered/bulleted fallback output."""
    text = (text or "").strip()
    if not text:
        return []
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            return _ordered_unique_text(x for x in parsed if isinstance(x, str))[:QUERY_DECOMPOSITION_MAX_SUBQUERIES]
    except Exception:
        pass

    # Try extracting the first JSON array if the model wrapped it in prose/code fences.
    match = re.search(r"\[[\s\S]*\]", text)
    if match:
        try:
            parsed = json.loads(match.group(0))
            if isinstance(parsed, list):
                return _ordered_unique_text(x for x in parsed if isinstance(x, str))[:QUERY_DECOMPOSITION_MAX_SUBQUERIES]
        except Exception:
            pass

    # Last-resort parser for numbered/bulleted lines.
    lines = [re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip(" \"'") for line in text.splitlines()]
    return _ordered_unique_text(lines)[:QUERY_DECOMPOSITION_MAX_SUBQUERIES]


def extract_query_intent(question: str) -> str:
    """Rewrite a raw benchmark question into a concise retrieval query via an
    OpenAI-compatible chat-completions call.

    Falls back to the original ``question`` when disabled, or on any client
    error, so retrieval always has *some* query to embed (never silently empty).
    """
    if not INTENT_EXTRACTION_ENABLED or _intent_client is None:
        return question
    try:
        response = _intent_client.chat.completions.create(
            model=INTENT_LLM_MODEL_NAME,
            messages=[
                {"role": "system", "content": INTENT_EXTRACTION_PROMPT},
                {"role": "user", "content": question},
            ],
            temperature=INTENT_LLM_TEMPERATURE,
        )
        cleaned = (response.choices[0].message.content or "").strip()
        return cleaned or question
    except Exception as exc:
        print(f"  [intent extraction failed for: {question[:60]!r}] {exc} — falling back to raw question")
        return question


def decompose_original_query(question: str) -> list[str]:
    """Split the original question into focused vector-retrieval subqueries.

    Uses the same OpenAI-compatible client/API key as intent extraction. Falls
    back to ``[question]`` when disabled or if the LLM call/parsing fails.
    """
    if not QUERY_DECOMPOSITION_ENABLED or _intent_client is None:
        return [question]
    try:
        response = _intent_client.chat.completions.create(
            model=INTENT_LLM_MODEL_NAME,
            messages=[
                {"role": "system", "content": QUERY_DECOMPOSITION_PROMPT},
                {"role": "user", "content": question},
            ],
            temperature=INTENT_LLM_TEMPERATURE,
        )
        content = response.choices[0].message.content or ""
        subqueries = _parse_query_list(content)
        return subqueries or [question]
    except Exception as exc:
        print(f"  [query decomposition failed for: {question[:60]!r}] {exc} — falling back to raw question")
        return [question]


def build_retrieval_queries(question: str) -> tuple[str, list[str]]:
    """Return (intent_query, retrieval_queries) for one benchmark question."""
    intent_query = extract_query_intent(question)
    if QUERY_DECOMPOSITION_ENABLED:
        queries = decompose_original_query(question)
        if QUERY_DECOMPOSITION_INCLUDE_INTENT_QUERY:
            queries = [intent_query, *queries]
        retrieval_queries = _ordered_unique_text(queries)
    else:
        retrieval_queries = [intent_query]
    return intent_query, retrieval_queries


# Quick sanity check on the first eligible case.
if eligibility_summary.eligible:
    _sample_question = eligibility_summary.eligible[0].question
    _sample_intent_query, _sample_retrieval_queries = build_retrieval_queries(_sample_question)
    print(f"Sample raw question     : {_sample_question}")
    print(f"Sample extracted query  : {_sample_intent_query}")
    print(f"Sample retrieval queries: {_sample_retrieval_queries}")



## 5c. Optional cross-encoder reranker (BAAI/bge-reranker-v2-m3)

This cell loads a second-stage cross-encoder reranker. It does **not** replace FAISS retrieval: FAISS first retrieves a wider candidate pool, optional RRF merges multi-query results, then the cross-encoder scores `(query, candidate_text)` pairs and reorders candidates before final top-k evaluation.


In [ ]:
_cross_encoder_reranker = None
_cross_encoder_device_resolved = "n/a"


def _resolve_torch_device(requested: str) -> str:
    requested = str(requested or "auto").strip().lower()
    if requested and requested != "auto":
        return requested
    try:
        import torch

        if torch.cuda.is_available():
            return "cuda"
        if getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
            return "mps"
    except Exception:
        pass
    return "cpu"


if CROSS_ENCODER_RERANK_ENABLED and not DEV_HASHING:
    from sentence_transformers import CrossEncoder

    _cross_encoder_device_resolved = _resolve_torch_device(CROSS_ENCODER_DEVICE)
    print(f"Loading cross-encoder reranker {CROSS_ENCODER_MODEL_NAME!r} on device={_cross_encoder_device_resolved!r}")
    _cross_encoder_reranker = CrossEncoder(
        CROSS_ENCODER_MODEL_NAME,
        device=_cross_encoder_device_resolved,
        max_length=CROSS_ENCODER_MAX_LENGTH,
    )
    print("Cross-encoder reranker ready.")
elif CROSS_ENCODER_RERANK_ENABLED and DEV_HASHING:
    print("Cross-encoder reranking requested but DEV_HASHING=True; reranker disabled for smoke-test mode.")
else:
    print("Cross-encoder reranking DISABLED.")


## 6. Retrieval + scoring loop (document/provision metrics)

`cases` is freshly re-initialized every time this cell runs, so reruns after a configuration change never mix stale per-case rows into the new aggregate.

The retriever still returns chunks, but scoring maps those chunks to their document/provision identifiers and reports only:

- `Doc_Precision`
- `Doc_Recall`
- `Provision_Precision`
- `Provision_Recall`


Provision IDs are normalized to their final unit number before scoring (for example, `183807::preamble::0` becomes `0`).


In [ ]:
from evaluation.retrieval_eval_report import RetrievalCaseResult

metric_keys = ["Doc_Precision", "Doc_Recall", "Provision_Precision", "Provision_Recall"]
cases: list[RetrievalCaseResult] = []  # re-initialized on every execution (FR-014)
extracted_queries: dict[str, str] = {}  # qa_id -> intent-extracted retrieval query
decomposed_queries_by_qa: dict[str, list[str]] = {}  # qa_id -> query-decomposition subqueries / retrieval queries
retrieved_document_ids_by_qa: dict[str, list[str]] = {}
retrieved_provision_ids_by_qa: dict[str, list[str]] = {}
ground_truth_document_ids_by_qa: dict[str, list[str]] = {}
ground_truth_provision_ids_by_qa: dict[str, list[str]] = {}
error_count = 0


def _ordered_unique(values):
    seen = set()
    out = []
    for value in values:
        if value and value not in seen:
            seen.add(value)
            out.append(value)
    return out


def _chunk_unique_key(chunk):
    return getattr(chunk, "chunk_id", None) or getattr(chunk, "id_str", None) or repr(chunk)


def _merge_retrieval_results(query_results, limit=None):
    """Merge retrieval result.chunk lists with Reciprocal Rank Fusion (RRF)."""
    limit = limit or max_k
    if not RRF_ENABLED:
        seen = set()
        merged = []
        for result in query_results:
            for chunk in getattr(result, "chunks", []):
                key = _chunk_unique_key(chunk)
                if key not in seen:
                    seen.add(key)
                    merged.append(chunk)
        return merged[:limit]

    ranked = {}
    first_seen = {}
    best_chunks = {}
    ordinal = 0
    for result_index, result in enumerate(query_results):
        for rank, chunk in enumerate(getattr(result, "chunks", []), start=1):
            key = _chunk_unique_key(chunk)
            if key not in first_seen:
                first_seen[key] = ordinal
                best_chunks[key] = chunk
                ordinal += 1
            ranked[key] = ranked.get(key, 0.0) + 1.0 / (RRF_K + rank)

    ordered_keys = sorted(ranked, key=lambda key: (-ranked[key], first_seen[key]))
    return [best_chunks[key] for key in ordered_keys[:limit]]


def _chunk_rerank_text(chunk):
    parts = [
        getattr(chunk, "title", ""),
        getattr(chunk, "citation_label", ""),
        getattr(chunk, "citation_anchor", ""),
        getattr(chunk, "chunk_text", ""),
    ]
    text = "\n".join(str(part).strip() for part in parts if str(part or "").strip())
    return text or repr(chunk)


def _cross_encoder_rerank_chunks(query: str, chunks: list, limit: int):
    if not CROSS_ENCODER_RERANK_ENABLED or _cross_encoder_reranker is None or not chunks:
        return chunks[:limit]
    pairs = [(query, _chunk_rerank_text(chunk)) for chunk in chunks]
    scores = _cross_encoder_reranker.predict(
        pairs,
        batch_size=CROSS_ENCODER_BATCH_SIZE,
        show_progress_bar=False,
    )
    scored = list(zip(chunks, [float(score) for score in scores]))
    for chunk, ce_score in scored:
        try:
            chunk.metadata = {**(getattr(chunk, "metadata", {}) or {}), "cross_encoder_score": ce_score}
        except Exception:
            pass
    scored.sort(key=lambda item: item[1], reverse=True)
    return [chunk for chunk, _ in scored[:limit]]


def _chunk_document_id(chunk):
    metadata = getattr(chunk, "metadata", {}) or {}
    return str(
        metadata.get("doc_id")
        or metadata.get("document_id")
        or getattr(chunk, "id_str", "")
        or ""
    )


def _provision_unit(value):
    """Use only the final unit number of a provision ID.

    Example: "183807::preamble::0" -> "0".
    """
    text = str(value or "")
    return text.rsplit("::", 1)[-1] if text else ""


def _chunk_provision_id(chunk):
    metadata = getattr(chunk, "metadata", {}) or {}
    return _provision_unit(
        metadata.get("unit_id")
        or metadata.get("provision_id")
        or getattr(chunk, "parent_unit_id", "")
    )


def _precision(retrieved_ids, ground_truth_ids):
    retrieved = set(retrieved_ids)
    ground_truth = set(ground_truth_ids)
    if not retrieved:
        return 0.0
    return len(retrieved & ground_truth) / len(retrieved)


def _recall(retrieved_ids, ground_truth_ids):
    retrieved = set(retrieved_ids)
    ground_truth = set(ground_truth_ids)
    if not ground_truth:
        return 0.0
    return len(retrieved & ground_truth) / len(ground_truth)


def build_doc_provision_metrics(retrieved_document_ids, retrieved_provision_ids, ground_truth_document_ids, ground_truth_provision_ids):
    return {
        "Doc_Precision": _precision(retrieved_document_ids, ground_truth_document_ids),
        "Doc_Recall": _recall(retrieved_document_ids, ground_truth_document_ids),
        "Provision_Precision": _precision(retrieved_provision_ids, ground_truth_provision_ids),
        "Provision_Recall": _recall(retrieved_provision_ids, ground_truth_provision_ids),
    }


for case in eligibility_summary.eligible:
    gt_document_ids = sorted(case.ground_truth_document_ids)
    gt_provision_ids = sorted(_provision_unit(pid) for pid in case.ground_truth_provision_ids if _provision_unit(pid))
    ground_truth_document_ids_by_qa[case.qa_id] = gt_document_ids
    ground_truth_provision_ids_by_qa[case.qa_id] = gt_provision_ids

    try:
        retrieval_query, retrieval_queries = build_retrieval_queries(case.question)
        extracted_queries[case.qa_id] = retrieval_query
        decomposed_queries_by_qa[case.qa_id] = retrieval_queries

        candidate_multiplier = max(1, RRF_CANDIDATE_MULTIPLIER)
        if CROSS_ENCODER_RERANK_ENABLED and _cross_encoder_reranker is not None:
            candidate_multiplier = max(candidate_multiplier, CROSS_ENCODER_CANDIDATE_MULTIPLIER)
        candidate_top_n = max_k * candidate_multiplier
        query_results = [retriever.retrieve(q, filter_profile="broad", top_k=max(candidate_top_n, config.top_k), top_n=candidate_top_n) for q in retrieval_queries]
        candidate_chunks = _merge_retrieval_results(query_results, limit=candidate_top_n)
        rerank_query = retrieval_query if QUERY_DECOMPOSITION_ENABLED else retrieval_queries[0]
        retrieved_chunks = _cross_encoder_rerank_chunks(rerank_query, candidate_chunks, max_k)

        retrieved_chunk_ids = [chunk.chunk_id for chunk in retrieved_chunks]
        retrieved_document_ids = _ordered_unique(_chunk_document_id(chunk) for chunk in retrieved_chunks)
        retrieved_provision_ids = _ordered_unique(_chunk_provision_id(chunk) for chunk in retrieved_chunks)
        retrieved_document_ids_by_qa[case.qa_id] = retrieved_document_ids
        retrieved_provision_ids_by_qa[case.qa_id] = retrieved_provision_ids

        metrics_row = build_doc_provision_metrics(
            retrieved_document_ids,
            retrieved_provision_ids,
            gt_document_ids,
            gt_provision_ids,
        )
        cases.append(
            RetrievalCaseResult(
                qa_id=case.qa_id,
                mode="vector_only",
                question=case.question,
                category=case.category,
                difficulty=case.difficulty,
                answer_type=case.answer_type,
                ground_truth_chunk_ids=[],
                retrieved_chunk_ids=retrieved_chunk_ids,
                metrics=metrics_row,
            )
        )
    except Exception as exc:  # surfaced per-case, does not abort the whole run
        error_count += 1
        decomposed_queries_by_qa[case.qa_id] = []
        retrieved_document_ids_by_qa[case.qa_id] = []
        retrieved_provision_ids_by_qa[case.qa_id] = []
        cases.append(
            RetrievalCaseResult(
                qa_id=case.qa_id,
                mode="vector_only",
                question=case.question,
                category=case.category,
                difficulty=case.difficulty,
                answer_type=case.answer_type,
                ground_truth_chunk_ids=[],
                retrieved_chunk_ids=[],
                metrics={key: 0.0 for key in metric_keys},
                error=str(exc),
            )
        )

print(f"Evaluated {len(cases)} case(s); error_count={error_count}")






## 7. Overall + breakdown metrics tables (FR-005)

In [ ]:
import pandas as pd

from evaluation.metrics import aggregate, aggregate_by

case_dicts = [{"category": c.category, "difficulty": c.difficulty, "answer_type": c.answer_type, **c.metrics} for c in cases]

overall = aggregate(case_dicts, metric_keys)
by_category = aggregate_by(case_dicts, "category", metric_keys)
by_difficulty = aggregate_by(case_dicts, "difficulty", metric_keys)
by_answer_type = aggregate_by(case_dicts, "answer_type", metric_keys)

print("=== VECTOR-ONLY retrieval evaluation — run summary ===")
print(f"total_rows                  = {eligibility_summary.total_rows}")
print(f"evaluated                   = {len(cases)}")
print(f"skipped_unanswerable         = {eligibility_summary.skipped_unanswerable}")
print(f"skipped_missing_ground_truth = {eligibility_summary.skipped_missing_ground_truth}")
print(f"error_count                  = {error_count}")

print("\nOverall metrics:")
display(pd.DataFrame([overall]))

print("\nBy category:")
display(pd.DataFrame.from_dict(by_category, orient="index"))

print("\nBy difficulty:")
display(pd.DataFrame.from_dict(by_difficulty, orient="index"))

print("\nBy answer_type:")
display(pd.DataFrame.from_dict(by_answer_type, orient="index"))


## 8. Per-case results (document/provision evaluation)


In [ ]:
per_case_rows = [
    {
        "qa_id": c.qa_id,
        "question": c.question,
        "retrieval_query": extracted_queries.get(c.qa_id, c.question),
        "retrieval_queries": decomposed_queries_by_qa.get(c.qa_id, [extracted_queries.get(c.qa_id, c.question)]),
        "ground_truth_document_ids": ground_truth_document_ids_by_qa.get(c.qa_id, []),
        "retrieved_document_ids": retrieved_document_ids_by_qa.get(c.qa_id, []),
        "ground_truth_provision_ids": ground_truth_provision_ids_by_qa.get(c.qa_id, []),
        "retrieved_provision_ids": retrieved_provision_ids_by_qa.get(c.qa_id, []),
        "error": c.error,
        **c.metrics,
    }
    for c in cases
]

per_case_df = pd.DataFrame(per_case_rows)
display(per_case_df)


## 9. Optional: persist artifacts (FR-008)

Writes `retrieval_cases.jsonl` and `retrieval_metrics.json` under `OUT_DIR` with document/provision metrics only. Skipped by default — set `PERSIST = True` below and re-run this cell to trigger it.


In [ ]:
from evaluation.retrieval_eval_report import ModeRunSummary, write_case_jsonl, write_metrics_json

PERSIST = False  # flip to True and re-run this cell to persist artifacts

if PERSIST:
    run_summary = ModeRunSummary(
        mode="vector_only",
        config={
            "index_dir": str(INDEX_DIR),
            "model": MODEL_NAME,
            "score_threshold": SCORE_THRESHOLD,
            "expand_units": EXPAND_UNITS,
            "top_n": max_k,
            "metrics": metric_keys,
            "sample_limit": SAMPLE_LIMIT,
            "dev_hashing": DEV_HASHING,
            "intent_extraction_enabled": INTENT_EXTRACTION_ENABLED,
            "query_decomposition_enabled": QUERY_DECOMPOSITION_ENABLED,
            "query_decomposition_include_intent_query": QUERY_DECOMPOSITION_INCLUDE_INTENT_QUERY,
            "query_decomposition_max_subqueries": QUERY_DECOMPOSITION_MAX_SUBQUERIES,
            "rrf_enabled": RRF_ENABLED,
            "rrf_k": RRF_K,
            "rrf_candidate_multiplier": RRF_CANDIDATE_MULTIPLIER,
            "cross_encoder_rerank_enabled": CROSS_ENCODER_RERANK_ENABLED,
            "cross_encoder_model_name": CROSS_ENCODER_MODEL_NAME,
            "cross_encoder_device": _cross_encoder_device_resolved,
            "cross_encoder_candidate_multiplier": CROSS_ENCODER_CANDIDATE_MULTIPLIER,
            "cross_encoder_batch_size": CROSS_ENCODER_BATCH_SIZE,
            "cross_encoder_max_length": CROSS_ENCODER_MAX_LENGTH,
        },
        total_rows=eligibility_summary.total_rows,
        evaluated=len(cases),
        skipped_unanswerable=eligibility_summary.skipped_unanswerable,
        skipped_missing_ground_truth=eligibility_summary.skipped_missing_ground_truth,
        error_count=error_count,
        overall=overall,
        by_category=by_category,
        by_difficulty=by_difficulty,
        by_answer_type=by_answer_type,
    )
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    write_case_jsonl(OUT_DIR / "retrieval_cases.jsonl", cases)
    write_metrics_json(OUT_DIR / "retrieval_metrics.json", run_summary, run_summary.config)
    print(f"Persisted artifacts under {OUT_DIR}")
else:
    print("PERSIST is False — nothing written. Set PERSIST = True above and re-run this cell to persist.")

